In [ ]:
# processing to combine across batches



import pandas as pd
import matplotlib.pylab as plt
import os
import importlib
import analysis_utils
import numpy as np
import json
import bisect

importlib.reload(analysis_utils)


def find_closest_index(sorted_list, target):
    idx = bisect.bisect_left(sorted_list, target)
    if idx == 0:
        return 0
    if idx == len(sorted_list):
        return len(sorted_list) - 1
    if target < sorted_list[idx]:
        return idx - 1
    else:
        return idx


data_dir = "../human-data/play-exp/human-v-human/raw-data/"

all_comments_cross_batch = {}

n_batches = 10

for batch_idx in range(n_batches):

    batch_idx += 1
    print("---------\n\n")
    print("ON BATCH:", batch_idx)
    print("\n\n---------\n\n")

    subdir = f"final-batch{batch_idx}"

    pth = f"{data_dir}{subdir}/"
    data_files = {f.split(".csv")[0]: pd.read_csv(f"{pth}{f}") for f in os.listdir(f"{pth}")}

    uid2prolific = {}
    uid2comments = {}
    arena2players = {}
    arena2judgement = {}

    for _, entry in data_files["player"].iterrows():
        dt = entry["endedLastChangedAt"]
        entered_id = entry.participantIdentifier
        uid = entry.id
        arena_id = entry.gameID
        comments = entry.exitSurvey

        if str(comments) == "nan" or len(eval(comments)) == 0:
            continue

        if len(entered_id) != 24 or str(arena_id) == "nan":
            continue
        uid2prolific[uid] = entered_id
        uid2comments[uid] = eval(comments)

        if arena_id not in arena2players:
            arena2players[arena_id] = {uid}
        else:
            arena2players[arena_id].add(uid)

        if "advantage" in entry["treatmentName"]:
            arena2judgement[arena_id] = "exp_utility"
        else:
            arena2judgement[arena_id] = "fun"

    all_comments_cross_batch[batch_idx] = uid2comments
    uids = sorted(list(uid2prolific.keys()))
    player2arena = {uid: arena_id for arena_id, players in arena2players.items() for uid in players}

    round_df = data_files["round"]
    arena2stimuli = {}
    arena2game_idx = {}

    for arena_id in arena2players:
        arena_df = round_df.loc[round_df.gameID == arena_id]

        stimuli = list(arena_df.stimuliID)

        sub_uids = arena2players[arena_id]
        if len(sub_uids) <= 1:
            for uid in sub_uids:
                uids.remove(uid)
            continue
        arena2stimuli[arena_id] = stimuli
        arena2game_idx[arena_id] = {stim: game_idx for game_idx, stim in enumerate(stimuli)}
    print(arena2stimuli)

    uid2judgements = {uid: {} for uid in uids}
    player_orderings = {uid: {} for uid in uids}
    slider_df = data_files["playerRound"]

    arena2players = {arena: set() for arena in arena2game_idx}

    for uid in uids:
        games_played = arena2stimuli[player2arena[uid]]
        judge_df = slider_df.loc[slider_df.playerID == uid].reset_index()
        for game_idx, entry in judge_df.iterrows():
            if entry["gameOutcome"] == "TBD":
                continue
            val = []

            stim_id = entry["stimuliID"]

            if "judgmentDraw" in entry and str(entry["judgmentDraw"]) != "nan":
                val_draw = entry["judgmentDraw"]
                val_firstplayer = entry["judgmentAdvantage"]
                val = [val_firstplayer, val_draw]
            else:
                if "judgment" in entry:
                    val = [entry["judgment"]]
                else:
                    val = ["nan", "nan"]

            if "judgmentSkill" in entry:
                val_skill = entry["judgmentSkill"]
            else:
                val_skill = -1

            val.append(val_skill)

            outcome = entry["gameOutcome"]
            order = entry["playerOrder"]

            player_orderings[uid][stim_id] = order
            arena = player2arena[uid]
            arena2players[arena].add(uid)
            stim_df = round_df.loc[round_df.stimuliID == stim_id]
            arena_df = stim_df.loc[stim_df.gameID == player2arena[uid]]
            player_orders = eval(arena_df["playerOrders"].iloc[0])
            if player_orders[uid] != order:
                print(stim_id, order, player_orders)

            uid2judgements[uid][stim_id] = {
                "val": val,
                "order": order,
                "outcome": outcome,
                "game_order": game_idx,
            }

    all_stimuli = list(set().union(*arena2stimuli.values()))

    all_stimuli_data = {stim: {
        "arena": [],
        "start": [],
        "duration": [],
        "move_times": [],
        "boards": [],
        "draw_requests": [],
        "draw_rejects": [],
        "draw_accepts": [],
        "draw_info": [],
        "outcome": [],
        "judgements": [],
        "player_judgements": [],
        "player_exp_payoffs": [],
        "order2player": [],
        "judgement_type": [],
        "surrender_info": [],
    } for stim in all_stimuli}

    main_fig_dir = "figs"

    win_judgements = []
    loss_judgements = []
    draw_judgements = []
    save_arena_figs = True

    stim2draw_accepts = {stim: 0 for stim in all_stimuli}

    def switch_player(player, players):
        return [p for p in players if player != p][0]

    for i, stim in enumerate(all_stimuli):
        stim_df = round_df.loc[round_df.stimuliID == stim]
        for j, entry in stim_df.iterrows():
            arena = entry.gameID
            if arena not in arena2players:
                continue

            judgement_type = arena2judgement[arena]

            start = entry.startLastChangedAt
            end = entry.endedLastChangedAt
            round_start = analysis_utils.parse_time(start)
            round_end = analysis_utils.parse_time(end)
            duration = round_end - round_start
            all_boards = entry.allBoards
            outcome = entry.gameWinner

            if str(entry.moveTimes) == "nan":
                print("ERROR")
                continue
            raw_move_times = eval(entry.moveTimes)
            if len(raw_move_times) == 0:
                print(arena, i, j)
            move_times = []
            move_start_time = 0
            for move_end_time in raw_move_times:
                move_time_seconds = (move_end_time - move_start_time) / 1000
                move_times.append(move_time_seconds)
                move_start_time = move_end_time

            surrenders_made = entry.surrenderMade
            if str(surrenders_made) != "nan":
                surrenders_made = eval(entry.surrenderMade)
                surrender_player, surrender_time = surrenders_made
                idx = find_closest_index(raw_move_times, surrender_time)
                if len(eval(all_boards)) != 0:
                    surrender_board = eval(all_boards)[idx]
                else:
                    surrender_board = None

                if "7.0*7.0" in stim and arena == "01JKXV9AKP0F550KZB23PTMDY6":
                    surrender_board = eval(all_boards)[-2]

                surrender_info = [surrender_player, surrender_time, surrender_board]
            else:
                surrender_info = None

            draws = eval(entry.allDrawAccepts)
            draw_requests = eval(entry.allDrawRequests)
            rejects = eval(entry.allDrawRejects)

            draw_info = {"reject": [], "accept": []}
            if len(draws) != 0:
                stim2draw_accepts[stim] += 1

                player_request, request_time = draw_requests[-1]
                idx = find_closest_index(raw_move_times, draw_requests[-1][1])
                if len(eval(all_boards)) == 0:
                    request_board = [[]]
                else:
                    request_board = eval(all_boards)[idx]
                    draw_info["accept"].append([player_request, request_time, request_board])

            if len(draws) == 0 and len(draw_requests) > 0:

                req_entries = [req_time for _, req_time in draw_requests]
                for reject_time in rejects:
                    req_idx = find_closest_index(req_entries, reject_time)
                    req_entry = draw_requests[req_idx]
                    player_request, request_time = req_entry
                    idx = find_closest_index(raw_move_times, request_time)
                    request_board = eval(all_boards)[idx]
                    draw_info["reject"].append([player_request, request_time, request_board])

                leftover_req = [
                    req_entry
                    for req_entry in draw_requests
                    if len(rejects) == 0
                    or np.any([
                        move_time > req_entry[1] and req_entry[1] > np.max(rejects)
                        for move_time in raw_move_times
                    ])
                ]

                if len(leftover_req) != 0:
                    for player_request, request_time in leftover_req:
                        idx = find_closest_index(raw_move_times, request_time)
                        request_board = eval(all_boards)[idx]
                        draw_info["reject"].append([player_request, request_time, request_board])

            n_rows, n_cols, win_conds = stim.split("*")
            n_rows = int(float(n_rows))
            n_cols = int(float(n_cols))
            if arena not in arena2game_idx:
                continue
            game_order = arena2game_idx[arena][stim]

            players = sorted(arena2players[arena])
            if len(players) == 0:
                continue

            colors = ["purple", "green"]
            player_colors = {player: colors[player_idx] for player_idx, player in enumerate(players)}
            order2player = {player_orderings[player][stim]: player for player in players}
            player2order = {v: k for k, v in order2player.items()}

            player_judgements = {player: uid2judgements[player][stim]["val"] for player in players}

            if outcome == 0.0:
                outcome = "Draw"
                vals = list(player_judgements.values())
                draw_judgements.extend(vals)
            else:
                outcome = order2player[outcome]
                win_judgements.append(player_judgements[outcome])
                loss_judgements.append(player_judgements[switch_player(outcome, players)])

            player_exp_payoffs = {}

            if judgement_type == "exp_utility":

                for player, (pfirst, pdraw, pskill) in player_judgements.items():
                    if str(pdraw) == "nan" or str(pskill) == "nan":
                        continue
                    pwin = analysis_utils.get_pwin(pdraw, pfirst)
                    pdraw /= 100
                    pwin /= 100
                    payoff = analysis_utils.compute_utility(pdraw, pwin)
                    player_exp_payoffs[player] = payoff

            try:
                player_move_map, move_order_map = analysis_utils.process_game_moves(all_boards)
                fig = analysis_utils.construct_board(
                    player_move_map, move_order_map, n_rows, n_cols, win_conds,
                    draw_accepted=len(draws) > 0,
                    player_colors=player_colors, player_orders=order2player,
                    player_judgements=player_judgements,
                    game_outcome=outcome,
                )
            except:
                print("error: ", arena)

            all_stimuli_data[stim]["arena"].append(arena)
            all_stimuli_data[stim]["boards"].append(all_boards)
            all_stimuli_data[stim]["outcome"].append(outcome)
            all_stimuli_data[stim]["duration"].append(duration)
            all_stimuli_data[stim]["move_times"].append(move_times)
            all_stimuli_data[stim]["judgements"].append(list(player_judgements.values()))
            all_stimuli_data[stim]["player_judgements"].append(player_judgements)
            all_stimuli_data[stim]["player_exp_payoffs"].append(player_exp_payoffs)
            all_stimuli_data[stim]["order2player"].append(order2player)
            all_stimuli_data[stim]["draw_requests"].append(draw_requests)
            all_stimuli_data[stim]["judgement_type"].append(judgement_type)
            all_stimuli_data[stim]["draw_accepts"].append(draws)
            all_stimuli_data[stim]["draw_info"].append(draw_info)
            all_stimuli_data[stim]["surrender_info"].append(surrender_info)

            if save_arena_figs:
                fig_dir = f"{main_fig_dir}/{arena}/"
                if not os.path.exists(fig_dir):
                    os.makedirs(fig_dir)
                fig.tight_layout()
                fig.savefig(f"{fig_dir}/game_idx_{game_order}_{stim}.png", dpi=400)
            plt.close()

    with open(
        f"/Users/katiecollins/intuitive-game-reasoning/human-data/play-exp/human-v-human/final-play/humanhuman_{subdir}.json",
        "w",
    ) as f:
        json.dump(all_stimuli_data, f)


In [ ]:
with open("gameplay_user_comments.json", "w") as f:
    json.dump(all_comments_cross_batch, f)
